# SuperNormal-RS — GPU Runner Notebook (A100 / L4 / G4 / T4)

Run this notebook on a **GPU Colab runtime** (`Runtime -> Change runtime type -> A100 / L4 / T4`).
It loads the prebuilt CUDA binary from Google Drive with **zero compilation time**, verifies CUDA parity with `doctor`, runs reconstruction on local SSD (`/content`), and copies the results back to Google Drive.

In [ ]:
# 1. Check GPU + Mount Google Drive + Copy prebuilt binary to fast local SSD
from google.colab import drive
import os, shutil

!nvidia-smi

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/supernormal_colab"
BIN_SRC = f"{DRIVE_DIR}/bin/supernormal"
BIN_LOCAL = "/content/supernormal"

assert os.path.isfile(BIN_SRC), f"Prebuilt binary not found at {BIN_SRC}. Run colab_build_supernormal_cpu.ipynb first!"
shutil.copy2(BIN_SRC, BIN_LOCAL)
os.chmod(BIN_LOCAL, 0o755)
print("Copied prebuilt binary to", BIN_LOCAL)

In [ ]:
# 2. Verify CUDA backend works cleanly on this Colab GPU (~3 seconds)
!/content/supernormal --api cuda doctor

In [ ]:
# 3. Copy dataset from Google Drive to fast local SSD (/content/data) and run training
import os, shutil

DATASET_NAME = "horse_960"  # Put your dataset folder inside MyDrive/supernormal_colab/datasets/
RUN_NAME = f"{DATASET_NAME}_colab_cuda"

DRIVE_DATASET = f"{DRIVE_DIR}/datasets/{DATASET_NAME}"
LOCAL_DATASET = f"/content/data/{DATASET_NAME}"
LOCAL_OUT = f"/content/experiments/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_DIR}/experiments/{RUN_NAME}"

if not os.path.exists(LOCAL_DATASET):
    assert os.path.isdir(DRIVE_DATASET), f"Dataset not found at {DRIVE_DATASET}"
    print(f"Copying {DRIVE_DATASET} -> {LOCAL_DATASET} for fast I/O...")
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)

os.makedirs(LOCAL_OUT, exist_ok=True)

# Uses the updated defaults: --steps 30000 --query 131072 --save-freq 30000 --mesh-freq 30000 --mesh-resolution 1024
!/content/supernormal --api cuda train --data "{LOCAL_DATASET}" --out "{LOCAL_OUT}"

In [ ]:
# 4. Persist results (mesh PLY, wall_time.json, metrics.csv, checkpoint) back to Google Drive
import shutil, os

if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)
print("Saved all outputs to:", DRIVE_OUT)
!cat "{LOCAL_OUT}/wall_time.json"